In [1]:
import json
from datetime import datetime, timedelta
import time

### Transform Chatgpt4 answers to QA file for fine-tuning

In [10]:
"""
this script is used for transforming responses in QA_gpt.json file
into llm conversations that are suitable for llama 3.1 fine-tuning.
"""

with open('files/indicator_guide_compact.txt', encoding='utf-8') as f:
    indicator_mapping = f.read()

today = datetime.today()

INDICATOR_GUIDE = indicator_mapping + f"""
Note: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly

When asked about economic data available above, use the get_fred_data function with the appropriate series_id to gain data to support your response.

Otherwise, directly answer the question.

For vague or unspecified request, ask for further clarification and explanation.

You will receive data from ALL tool calls.

IMPORTANT:
1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like "-2y"
2. If no time period specified, use recent 1 year by default
3. Each series_id should only be called ONCE with a single continuous date range.
   - WRONG: calling GDP twice with 2018-2020 and 2021-2023
   - RIGHT: calling GDP once with 2018-2023
4. Today is {today.strftime('%Y-%m-%d')}
"""

def QA_gpt_transformer(file_path='QA_finetune3.json'):

  with open(file_path, encoding='utf-8') as f:
      file = json.load(f)

  results = []
  for idx, q in enumerate(file):
      question = q['question']
      tool_calls = q.get('tool_calls', [])
      final_answer = q.get('final_answer', "")

      # questions require tool calls
      if tool_calls:
        api_results = q['api_results']
        template = {
            "conversations": [
                {
                    "role": "system",
                    "content": f"You are an economic data assistant with access to FRED API. {INDICATOR_GUIDE}"
                },
                {
                    "role": "user",
                    "content": question
                },
            ]
        }

        # add tool call response from assistant
        tools = [[tool_call['series_id'], tool_call['start_date'], tool_call['end_date']] for tool_call in tool_calls]

        tool_calls_content = "\n".join([
            json.dumps({"name": "get_fred_data", "arguments": {"series_id": tool[0], "start_date": tool[1], "end_date": tool[2]}})
            for tool in tools])

        template['conversations'].append({
            "role": "assistant",
            "content": tool_calls_content
        })

        # add tool response from tool functions
        template['conversations'].append({
            "role": "tool",
            "content": json.dumps(api_results)
        })

        # add final response from assistant
        template['conversations'].append({
            "role": "assistant",
            "content": final_answer
        })

      # questions not require tool calls
      else:
        template = {
            "conversations": [
                {
                    "role": "system",
                    "content": f"You are an economic data assistant with access to FRED API. {INDICATOR_GUIDE}"
                },
                {
                    "role": "user",
                    "content": question
                },
                {
                    "role": "assistant",
                    "content": final_answer
                }
            ]
        }
      results.append(template)

  return results

In [11]:
results = QA_gpt_transformer(file_path='data/QA_finetune3.json')

# save file
with open('data/QA_conv3.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

### Create QA file for summary fine-tuning only 

#### QA structure after processing:

    system  → prompt
    user    → question
    user    → [Tool calls made]: {...} 
    tool    → FRED API data 
    assistant → summary       ← will only be fine-tuned on this


In [4]:
def prepare_summary_finetune(input_path, output_path):
    with open(input_path, encoding="utf-8") as f:
        data = json.load(f)
    
    SUMMARY_SYSTEM_PROMPT = """You are an economic data assistant. 
When provided with FRED economic data, analyze it clearly and concisely.
Focus on trends, key statistics, and answering the user's specific question."""
    
    processed = []
    skipped = 0
    
    for conv in data:
        messages = conv["conversations"]
        new_messages = []
        has_summary = False
        
        for msg in messages:
            if msg["role"] == "system":
                new_messages.append({
                    "role": "system",
                    "content": SUMMARY_SYSTEM_PROMPT
                })
            elif msg["role"] == "assistant":
                is_tool_call = msg["content"].strip().startswith("{\"name\"")
                if is_tool_call:
                    # tool call will be turned into user turn, loss will not be computed
                    new_messages.append({
                        "role": "user",
                        "content": f"[Tool calls made]: {msg['content']}"
                    })
                else:
                    # keep summary turn only
                    new_messages.append(msg)
                    has_summary = True
            else:
                new_messages.append(msg)
        
        if has_summary:
            processed.append({"conversations": new_messages})
        else:
            skipped += 1
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, ensure_ascii=False, indent=2)
    
    print(f"Processed: {len(processed)}, Skipped (no summary): {skipped}")
    print(f"Saved to {output_path}")


In [13]:
prepare_summary_finetune("data/QA_conv3.json", "data/QA_conv4.json")

Processed: 727, Skipped (no summary): 0
Saved to data/QA_conv4.json


### Transform QA into SFT format (available for LLaMA & GPT)

1. Remove:
- The second user (tool call)
- Tool role

2. Merge the tool data into the user
new_user_content = f“”"
{question}

Here is the economic data:
{compressed_tool_data}
“”"

3. Compress the tool data (very important)

In [9]:
import json

INPUT_FILE = "data/QA_conv4_rewrite.json"
OUTPUT_FILE = "data/QA_conv4_rewrite_cleaned.jsonl"


def extract_fields(conversations):
    question = None
    tool_content = None
    answer = None
    system_prompt = None

    for msg in conversations:
        role = msg.get("role")
        content = msg.get("content", "")

        if role == "system" and system_prompt is None:
            system_prompt = content

        elif role == "user":
            if "[Tool calls made]" not in content and question is None:
                question = content

        elif role == "tool" and tool_content is None:
            tool_content = content

        elif role == "assistant":
            answer = content

    return system_prompt, question, tool_content, answer


def compress_tool_content(tool_content_str):
    try:
        data = json.loads(tool_content_str)[0]

        success = data.get("success", True)

        if not success:
            return json.dumps({
                "success": False,
                "error": data.get("error", "unknown error")
            }, ensure_ascii=False)

        analysis = data.get("analysis", {})
        overview = analysis.get("overview", {})
        stats = analysis.get("key_statistics", {})
        trend = analysis.get("trend_analysis", {})

        compressed = {
            "success": True,
            "indicator": data.get("indicator_name"),
            "units": data.get("units"),
            "latest_value": overview.get("latest_value"),
            "average": stats.get("average"),
            "trend": trend.get("overall_trend"),
            "change": trend.get("total_change")
        }

        return json.dumps(compressed, ensure_ascii=False)

    except Exception:
        return json.dumps({
            "success": False,
            "error": "parse_error"
        })


def build_user_prompt(question, tool_data):
    return f"""{question}

Tool result:
{tool_data}
"""


def main():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    count = 0

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        for i, item in enumerate(data):
            conversations = item.get("conversations", [])

            system_prompt, question, tool_content, answer = extract_fields(conversations)

            if not question or  not answer:
                print(f"⚠️ Skipping item {i}")
                continue
            
            compressed_tool = compress_tool_content(tool_content)
            
            user_prompt = build_user_prompt(question, compressed_tool)

            messages = []

            if system_prompt:
                messages.append({
                    "role": "system",
                    "content": system_prompt
                })

            messages.append({
                "role": "user",
                "content": user_prompt
            })

            messages.append({
                "role": "assistant",
                "content": answer
            })

            out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
            count += 1

    print(f"\nDone! Total samples: {count}")
    print(f"Saved to: {OUTPUT_FILE}")


In [10]:
main()


Done! Total samples: 727
Saved to: data/QA_conv4_rewrite_cleaned.jsonl
